<a href="https://colab.research.google.com/github/ihsann2410010447/statistika-probabilitas-uniska/blob/main/Modul_6_Praktikum_Reliability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Modul 6 — Praktikum Distribusi Kontinu & MLOps Reliability

### Studi Kasus: Simulated ML API Latency + Lung Cancer Survival

**Mata Kuliah:** Statistika & Probabilitas (TIF1501)
**Program Studi:** Teknik Informatika (55201) · FTI · Uniska MAB Banjarmasin
**Semester:** 5 · 3 SKS

---

## 🎯 Tujuan Pembelajaran

1. **Memahami** distribusi Uniform U(a,b) dan aplikasinya di random sampling.
2. **Memodelkan** waktu antar kejadian dengan Exponential Exp(λ) + sifat memoryless.
3. **Menerapkan** metric MLOps reliability (p50/p95/p99, SLO, MTBF) di production.
4. **Melakukan** Monte Carlo simulation untuk estimasi dan uncertainty.

---

## 📋 Alur Notebook

| Bagian | Isi | Waktu |
|--------|-----|-------|
| **A** | Setup & Import | 5 menit |
| **B** | Uniform U(0,1) Basics | 10 menit |
| **C** | Monte Carlo: Estimasi π | 10 menit |
| **D** | Simulated ML API Latency Exp(λ) | 15 menit |
| **E** | Verifikasi Sifat Memoryless | 10 menit |
| **F** | Real-world: Lung Cancer Survival | 10 menit |
| **G** | SLO Analysis p50/p95/p99 | 15 menit |
| **H** | MTBF & Reliability + Refleksi | 15 menit |

**Total: ± 90 menit**

---

## 💡 Dual-Dataset Strategi

**Dataset 1 — Simulated ML API Latency (synthetic)**: Clean Exponential dengan λ terkontrol, cocok untuk belajar teori murni.

**Dataset 2 — Lung Cancer Survival (Kaggle)**: 228 pasien real-world, menunjukkan bagaimana data nyata APPROXIMATE Exp tapi tidak sempurna. Lebih realistic untuk analisis.


---

# 🔧 A. Setup & Import


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import uniform, expon, probplot

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Library siap untuk Modul 6!")
print(f"NumPy  : {np.__version__}")
print(f"SciPy  : loaded")


---

# 🎲 B. Uniform U(0,1) Basics

**Tujuan Pembelajaran 1** — Generate dan analisis variabel acak Uniform.


In [ ]:
# Generate 10.000 sample U(0, 1)
N = 10_000
u_samples = np.random.uniform(0, 1, N)

# Verifikasi sifat teoritis
E_theory = 0.5
Var_theory = 1/12

print(f"Teori U(0, 1):")
print(f"  E(X)   = (a+b)/2 = {E_theory}")
print(f"  Var(X) = (b-a)²/12 = {Var_theory:.4f}")
print(f"  σ      = √(1/12)  = {np.sqrt(Var_theory):.4f}")

print(f"\nEmpiris (n={N}):")
print(f"  Mean  = {u_samples.mean():.4f}")
print(f"  Var   = {u_samples.var():.4f}")
print(f"  Std   = {u_samples.std():.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(u_samples, bins=40, density=True, color='#14B8A6', edgecolor='black', alpha=0.75)
axes[0].axhline(1.0, color='red', linestyle='--', linewidth=2, label='Teori f(x) = 1')
axes[0].set_xlabel('x')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Histogram U(0,1) — n={N}')
axes[0].legend()

# QQ-plot vs uniform teoritis
probplot(u_samples, dist='uniform', plot=axes[1])
axes[1].set_title('QQ-Plot vs Uniform Teoritis')
plt.tight_layout()
plt.show()


---

# 🎯 C. Monte Carlo: Estimasi π dengan Uniform

**Tujuan Pembelajaran 4** — Prinsip Monte Carlo untuk estimasi numerik.

**Ide**: Tembak dart acak ke square [0,1]². Fraksi yang jatuh di quarter circle × 4 → estimasi π.


In [ ]:
def estimate_pi(N):
    x = np.random.uniform(0, 1, N)
    y = np.random.uniform(0, 1, N)
    inside = (x**2 + y**2) <= 1
    return 4 * inside.mean(), x, y, inside

# Konvergensi: coba berbagai N
Ns = [100, 1_000, 10_000, 100_000, 1_000_000]
print(f"{'N':>10} | {'π estimate':>14} | {'error':>12}")
print("-" * 42)
for N in Ns:
    pi_est, _, _, _ = estimate_pi(N)
    err = abs(pi_est - np.pi)
    print(f"{N:>10,} | {pi_est:>14.6f} | {err:>12.6f}")

print(f"\nπ sejati = {np.pi:.6f}")
print("💡 Error ≈ 1/√N — klasik rate konvergensi Monte Carlo!")


In [ ]:
# Visualisasi: dart yang kena vs tidak
pi_est, x, y, inside = estimate_pi(5000)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(x[inside], y[inside], s=4, c='#14B8A6', alpha=0.6, label=f'Hit ({inside.sum()})')
ax.scatter(x[~inside], y[~inside], s=4, c='#EA580C', alpha=0.4, label=f'Miss ({(~inside).sum()})')

# Draw quarter circle
theta = np.linspace(0, np.pi/2, 200)
ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.set_title(f'Monte Carlo π estimate: {pi_est:.4f} (N=5000)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


---

# 📡 D. Simulated ML API Latency: Exp(λ)

**Tujuan Pembelajaran 2** — Model latency inference ML API dengan Exponential.

**Skenario**: Backend ML API Tokopedia mengembalikan prediction dengan rata-rata latency 120ms. Latency individu bervariasi mengikuti Exponential.


In [ ]:
# Parameter: λ = 1/mean_latency, mean = 120ms
mean_latency = 120  # ms
lam = 1 / mean_latency

# Generate 10.000 request latencies
N = 10_000
latencies = np.random.exponential(scale=mean_latency, size=N)

print(f"Simulasi ML API Latency (n={N}):")
print(f"  λ (rate)   = {lam:.6f} per ms")
print(f"  1/λ (mean) = {1/lam:.2f} ms  (expected mean)")
print(f"\nStatistik empiris:")
print(f"  Mean = {latencies.mean():.2f} ms")
print(f"  Std  = {latencies.std():.2f} ms   (Exp: mean ≈ std)")
print(f"  Max  = {latencies.max():.2f} ms   (long tail!)")

# Fit visual
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(latencies, bins=80, density=True, color='#EA580C', edgecolor='black', alpha=0.7, label='Empiris')
x_range = np.linspace(0, latencies.max(), 300)
ax.plot(x_range, expon.pdf(x_range, scale=mean_latency), 'b-', linewidth=2.5,
        label=f'Exp(λ=1/{mean_latency}) teoritis')
ax.axvline(mean_latency, color='red', linestyle='--', linewidth=2, label=f'E(X)=1/λ={mean_latency}ms')
ax.set_xlabel('Latency (ms)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('ML API Latency: Fit Exp(λ) pada data simulasi', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


---

# 🧠 E. Verifikasi Sifat Memoryless

**Tujuan Pembelajaran 2** — Buktikan secara empiris: P(X > s+t | X > s) = P(X > t).


In [ ]:
# Test memoryless: pilih s=100ms, t=50ms
s, t = 100, 50

# Empirical conditional probability
cond_mask = latencies > s
X_given_s = latencies[cond_mask]  # subset yang sudah > s
P_cond_empirical = (X_given_s > (s + t)).mean()

# Marginal probability
P_marginal_empirical = (latencies > t).mean()

# Teoritis
P_cond_theory = np.exp(-lam * t)
P_marginal_theory = np.exp(-lam * t)

print(f"Verifikasi Memoryless Property (s={s}ms, t={t}ms):")
print()
print(f"P(X > s+t | X > s) empiris  = {P_cond_empirical:.4f}")
print(f"P(X > t) empiris            = {P_marginal_empirical:.4f}")
print(f"P(X > t) teoritis = e^(-λt) = {P_marginal_theory:.4f}")
print()
diff = abs(P_cond_empirical - P_marginal_empirical)
print(f"Selisih |P_cond - P_marginal| = {diff:.4f}  (should be ≈ 0)")
if diff < 0.02:
    print("\n✅ Memoryless property TERVERIFIKASI secara empiris!")
    print("   Server yang sudah hidup 100ms sama 'siap mati' dengan server baru.")


---

# 🏥 F. Real-world: Lung Cancer Survival

**Tujuan Pembelajaran 2** — Fit Exp pada data real-world; identifikasi deviasi.


In [ ]:
# Load lung cancer survival dataset (228 patients)
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/survival/cancer.csv"
cancer_df = pd.read_csv(url)
print(f"Dataset: {cancer_df.shape[0]} pasien, {cancer_df.shape[1]} kolom")
print(f"\nSurvival time (hari):")
print(cancer_df['time'].describe().round(2))

# Fit Exp via MLE
time_data = cancer_df['time'].values
lam_cancer = 1 / time_data.mean()
print(f"\nMLE λ̂ = 1/mean = {lam_cancer:.6f}")
print(f"MTBF (expected mean) = 1/λ̂ = {1/lam_cancer:.2f} hari")

# Compare empiris vs Exp fit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Histogram + Exp overlay
axes[0].hist(time_data, bins=30, density=True, color='#8B5CF6', edgecolor='black', alpha=0.7, label='Empiris')
x_range = np.linspace(0, time_data.max(), 300)
axes[0].plot(x_range, expon.pdf(x_range, scale=1/lam_cancer), 'r-', linewidth=2.5, label=f'Exp(λ={lam_cancer:.4f}) fit')
axes[0].set_xlabel('Survival time (hari)')
axes[0].set_ylabel('Density')
axes[0].set_title('Lung Cancer Survival: Fit Exp')
axes[0].legend()

# QQ-plot
probplot(time_data, dist=expon, sparams=(0, 1/lam_cancer), plot=axes[1])
axes[1].set_title('QQ-Plot vs Exponential')
plt.tight_layout()
plt.show()

print("\n💡 Real-world data MENDEKATI Exp tapi tidak sempurna.")
print("   Deviasi di tail → distribusi Weibull/Gamma mungkin lebih cocok.")


---

# 📊 G. SLO Analysis p50, p95, p99

**Tujuan Pembelajaran 3** — Hitung percentile untuk Service Level Objective (SLO).


In [ ]:
# Hitung percentile dari latency simulasi
p50 = np.percentile(latencies, 50)
p95 = np.percentile(latencies, 95)
p99 = np.percentile(latencies, 99)

print(f"=== LATENCY PERCENTILES (ML API) ===")
print(f"  p50 (median)      = {p50:.2f} ms")
print(f"  p95               = {p95:.2f} ms")
print(f"  p99               = {p99:.2f} ms")
print(f"  max observed      = {latencies.max():.2f} ms")

# Teoritis dari Exp: F(x) = 1 - e^(-λx) = p → x = -ln(1-p)/λ
p50_theory = -np.log(1-0.50) / lam
p95_theory = -np.log(1-0.95) / lam
p99_theory = -np.log(1-0.99) / lam
print(f"\n=== TEORITIS (Exp): x = -ln(1-p)/λ ===")
print(f"  p50 teori = {p50_theory:.2f} ms")
print(f"  p95 teori = {p95_theory:.2f} ms")
print(f"  p99 teori = {p99_theory:.2f} ms")

# SLO analysis
SLO = 300  # ms
violation_rate = (latencies > SLO).mean()
expected_rate_theory = np.exp(-lam * SLO)
print(f"\n=== SLO ANALYSIS (threshold {SLO}ms) ===")
print(f"  Violation rate empiris = {violation_rate*100:.2f}%")
print(f"  Expected (teori)       = {expected_rate_theory*100:.2f}%")

if violation_rate < 0.05:
    print(f"  ✅ Memenuhi SLO '95% < {SLO}ms'")
else:
    print(f"  🚨 TIDAK memenuhi SLO '95% < {SLO}ms' — perlu optimasi!")


In [ ]:
# Visualize SLO analysis
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(latencies, bins=100, color='#EA580C', edgecolor='black', alpha=0.7)

# Mark percentiles
for p_val, p_name, color in [(p50, 'p50', '#10B981'), (p95, 'p95', '#F59E0B'), (p99, 'p99', '#F43F5E')]:
    ax.axvline(p_val, color=color, linestyle='--', linewidth=2.5, label=f'{p_name} = {p_val:.1f}ms')

# SLO threshold
ax.axvline(SLO, color='black', linestyle=':', linewidth=2, label=f'SLO = {SLO}ms')

# Highlight violation area
ax.axvspan(SLO, latencies.max(), alpha=0.15, color='red', label=f'Violation ({violation_rate*100:.1f}%)')

ax.set_xlabel('Latency (ms)', fontsize=12)
ax.set_ylabel('Request count', fontsize=12)
ax.set_title('Distribusi Latency dengan Percentile + SLO Threshold', fontsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(0, 800)
plt.tight_layout()
plt.show()


---

# ⚙️ H. MTBF & Reliability + Refleksi

**Tujuan Pembelajaran 3** — MTBF untuk server uptime projection.

## H.1 MTBF (Mean Time Between Failures)


In [ ]:
# Skenario: Server ML crash setiap rata-rata 72 jam (MTBF)
MTBF = 72  # jam
lam_server = 1 / MTBF  # failures per hour

print(f"Server ML Reliability Model:")
print(f"  MTBF         = {MTBF} jam")
print(f"  λ (rate)     = {lam_server:.6f} per jam")
print(f"\nProbabilitas crash dalam periode t:")
for t in [1, 24, 72, 168, 720]:
    p_crash = 1 - np.exp(-lam_server * t)
    label = {1:'1 jam', 24:'1 hari', 72:'3 hari (= MTBF)', 168:'1 minggu', 720:'1 bulan'}[t]
    print(f"  P(crash dalam {label:<18}) = {p_crash*100:.2f}%")

# Expected failures per month
expected_failures_month = 720 / MTBF
print(f"\nEkspektasi crash per bulan = 720/{MTBF} = {expected_failures_month:.2f} kali")

# Uptime projection
uptime_pct_month = np.exp(-lam_server * 720) * 100
print(f"Uptime projection 1 bulan tanpa restart = {uptime_pct_month:.4f}%")
print(f"\n💡 Butuh auto-restart/redundancy untuk SLA 99.9% uptime!")


## H.2 Pertanyaan Refleksi

Jawab 5 pertanyaan berikut di cell markdown di bawah:

1. **Uniform vs Normal**: Kenapa U(0,1) umumnya dipakai sebagai dasar random number generator di library (np.random, random)? Bisakah kita generate Normal dari U(0,1)?

2. **Monte Carlo**: Pada estimasi π, kenapa error turun ~1/\sqrt{N} (bukan 1/N)? Apa implikasi: butuh 100× sample untuk 10× lebih akurat?

3. **Memoryless**: Jelaskan satu konsekuensi real-world dari sifat memoryless pada distribusi failure. Apakah sebenarnya realistic untuk hardware sungguhan?

4. **Percentile p99**: Kenapa Netflix/Shopee lebih peduli p99 dibanding mean latency? Berikan contoh dampak jika mereka hanya optimasi mean.

5. **MTBF & SLA**: Dengan MTBF 72 jam, apakah mungkin mencapai SLA 99.9% uptime bulanan tanpa redundancy? Strategi arsitektur apa yang perlu?

### ✏️ Jawaban Anda:

**1. Uniform vs Normal**
* **Mengapa $U(0,1)$ menjadi dasar:** Distribusi Uniform $U(0,1)$ sangat mudah diproduksi secara komputasi menggunakan algoritma deterministik (seperti *Pseudo-Random Number Generator* / PRNG, contohnya *Mersenne Twister*). $U(0,1)$ juga memiliki probabilitas yang sama di setiap titiknya, menjadikannya kanvas dasar yang ideal.
* **Menghasilkan Normal dari $U(0,1)$:** **Bisa.** Kita dapat menggunakan metode matematika seperti **Inverse Transform Sampling** menggunakan fungsi CDF terbalik dari distribusi normal (Probit), atau menggunakan algoritma yang sangat populer seperti **Box-Muller Transform** yang mengubah dua variabel acak independen $U(0,1)$ menjadi dua variabel acak normal standar independen.

**2. Monte Carlo (Konvergensi Error)**
* **Kenapa error turun $\approx 1/\sqrt{N}$:** Berdasarkan **Teorema Limit Pusat (Central Limit Theorem)** dan **Hukum Bilangan Besar (Law of Large Numbers)**, deviasi standar (standard error) dari estimasi rata-rata sampel berbanding terbalik dengan akar kuadrat dari jumlah sampel ($1/\sqrt{N}$).
* **Implikasi:** Ini berarti peningkatan akurasi memiliki hubungan non-linear yang melambat (hukum pengembalian yang berkurang). Untuk menaikkan akurasi sebesar 10 kali lipat, kita tidak bisa hanya menambah 10 kali sampel, melainkan harus menaikkan ukuran sampel sebanyak $10^2 = 100$ kali lipat. Ini membuat simulasi Monte Carlo murni sangat mahal secara komputasi untuk akurasi yang sangat tinggi.

**3. Memoryless Property**
* **Konsekuensi Real-World:** Sifat *memoryless* (tanpa memori) pada distribusi Eksponensial berarti bahwa probabilitas suatu komponen bertahan hidup selama $t$ waktu ke depan tidak bergantung pada seberapa lama komponen tersebut telah berjalan ($P(X > s+t \mid X > s) = P(X > t)$). Dengan kata lain, tidak ada efek "keausan" (wear-out).
* **Apakah realistis untuk hardware:** **Tidak realistis** untuk jangka panjang. Hardware fisik (seperti SSD, kipas pendingin, atau power supply) memiliki degradasi mekanis dan termal seiring waktu, sehingga tingkat kegagalannya meningkat (mengikuti distribusi Weibull dengan parameter bentuk $\beta > 1$ atau kurva bak mandi / *bathtub curve*). Sifat *memoryless* hanya realistis untuk memodelkan kegagalan acak eksternal (seperti lonjakan listrik tiba-tiba atau bencana alam) yang probabilitas kejadiannya konstan dari hari ke hari.

**4. Percentile p99 vs Mean Latency**
* **Mengapa peduli p99:** Mean (rata-rata) sering kali menyembunyikan masalah buruk karena tereduksi oleh mayoritas request yang cepat. Di platform skala besar seperti Netflix atau Shopee, p99 berarti 1 dari 100 pengguna (atau ribuan pengguna aktif per detik) mengalami delay yang sangat parah. Pengguna yang terkena p99 ini biasanya adalah pengguna aktif yang membawa banyak item belanjaan di keranjang mereka (menyebabkan proses query database lebih berat), sehingga jika dibiarkan lambat, mereka akan meninggalkan aplikasi (*churn*).
* **Dampak optimasi mean saja:** Jika sistem hanya mengoptimalkan rata-rata latency, tim engineering mungkin merasa puas melihat angka rata-rata yang turun dari 120ms ke 100ms. Namun, mereka akan melewatkan fakta bahwa p99 mereka tetap berada di 1500ms (1.5 detik), menyebabkan transaksi dari segmen pengguna paling potensial tetap mengalami kegagalan/timeout.

**5. MTBF & SLA Uptime**
* **Apakah mungkin tanpa redundancy:** **Sangat tidak mungkin.** Dengan MTBF 72 jam, rata-rata server akan crash setiap 3 hari sekali. Tanpa redundansi, jika proses restart dan recovery membutuhkan waktu minimal 5-10 menit saja, akumulasi downtime bulanan akan jauh melampaui batas SLA 99.9% (yang hanya memperbolehkan downtime maksimal $\approx 43.8$ menit dalam sebulan). Terlebih lagi, proyeksi uptime tanpa restart sama sekali selama sebulan adalah $0.0045\%$ (hampir mustahil bertahan tanpa crash).
* **Strategi Arsitektur yang Diperlukan:**
  * **Multi-Instance Redundancy (High Availability):** Menjalankan beberapa replika API di belakang sebuah **Load Balancer**, sehingga jika satu instance crash, traffic langsung dialihkan ke instance lain tanpa interupsi bagi pengguna.
  * **Auto-healing / Orchestration:** Menggunakan container orchestrator seperti **Kubernetes** untuk secara otomatis mendeteksi kegagalan pod (*Liveness Probe*) dan melakukan *auto-restart* dalam hitungan detik.
  * **Graceful Degradation / Circuit Breaker:** Menyiapkan fallback mechanism jika model ML gagal merespons, misalnya mengembalikan rekomendasi default (static/cached) agar sistem tidak langsung crash total.

## H.3 Checklist Submit

- [ ] Semua cell code berhasil dijalankan tanpa error
- [ ] Monte Carlo π konvergen ke 3.14 dengan N besar
- [ ] Fit Exp pada simulated latency mencocokkan teoritis
- [ ] Memoryless property TERVERIFIKASI (selisih < 0.02)
- [ ] Lung cancer QQ-plot menunjukkan deviasi dari Exp murni
- [ ] p50/p95/p99 dihitung + visualisasi SLO threshold
- [ ] MTBF projection untuk 1 bulan ditampilkan
- [ ] 5 pertanyaan refleksi dijawab
- [ ] Notebook di-rename: `M6-Praktikum-Reliability-NamaAnda.ipynb`
- [ ] Upload ke GitHub repository Anda
- [ ] Link GitHub + README dikumpulkan via LMS Uniska

## H.4 Push ke GitHub

1. Buka repository `statistika-probabilitas-uniska` di GitHub.
2. Klik folder `notebooks/` → **Add file → Upload files**.
3. Drag & drop notebook.
4. Commit message: `"Add Modul 6: Uniform, Eksponensial & MLOps Reliability"`.

---

## 🎓 Selamat! Anda Berhasil...

✅ Generate dan analisis variabel Uniform
✅ Mengestimasi π dengan Monte Carlo
✅ Model ML API latency dengan Exp(λ)
✅ Membuktikan sifat memoryless secara empiris
✅ Hitung p50/p95/p99 untuk SLO analysis
✅ Project MTBF untuk uptime SLA

## 📚 Bacaan Lanjutan

- **Bruce, P.** (2020). *Practical Statistics for Data Scientists*, Ch. 2.10-2.11 (Continuous Distributions).
- **Beyer, B. et al.** (Google SRE Book) — *Service Level Objectives* chapter.
- **StatQuest** YouTube — "Exponential Distribution Clearly Explained".
- **Metropolis & Ulam** (1949) — Original Monte Carlo paper.

## 🔮 Persiapan Modul 8

Modul 7 memasuki **CPMK072 — Analisis Data Terstruktur**:
- Measures of central tendency & dispersion
- Z-score & StandardScaler, Plotly
- **Milestone 1 Projek Besar**: EDA dataset Kaggle pilihan sendiri

**Tugas Mandiri sebelum M7:**
1. Selesaikan Modul-6.ipynb, push ke GitHub
2. Baca StatQuest "Normal Distribution"
3. Baca Bruce Ch. 1 (EDA fundamentals)

---

*Notebook by: Prodi Teknik Informatika · Uniska MAB Banjarmasin · Edisi 2026*
